# Practical 8: Train, Fine-Tune, and Deploy Open-Source Generative AI Models using Hugging Face

## Aim
To train, fine-tune, and deploy open-source generative AI models using Hugging Face.

In [ ]:
%pip install -q transformers datasets evaluate accelerate gradio huggingface_hub

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, pipeline
)
import numpy as np
import evaluate

print('Libraries imported successfully.')

## Step 1: Load Dataset and Tokenizer

In [ ]:
dataset = load_dataset('ag_news')

small_train = dataset['train'].shuffle(seed=42).select(range(2000))
small_test = dataset['test'].shuffle(seed=42).select(range(500))

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_tok = small_train.map(tokenize, batched=True)
test_tok = small_test.map(tokenize, batched=True)

train_tok = train_tok.rename_column('label', 'labels')
test_tok = test_tok.rename_column('label', 'labels')
train_tok.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tok.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print('Dataset prepared for fine-tuning.')

## Step 2: Fine-Tune Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir='hf_finetune_output',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
eval_results = trainer.evaluate()
print('Evaluation:', eval_results)

## Step 3: Save and Run Inference

In [ ]:
save_dir = 'hf_agnews_model'
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

classifier = pipeline('text-classification', model=save_dir, tokenizer=save_dir)
sample_text = 'The stock market gained after the central bank announcement.'
pred = classifier(sample_text)

print('Sample text:', sample_text)
print('Prediction:', pred)

## Step 4: Lightweight Deployment with Gradio

In [ ]:
import gradio as gr

label_map = {
    'LABEL_0': 'World',
    'LABEL_1': 'Sports',
    'LABEL_2': 'Business',
    'LABEL_3': 'Sci/Tech'
}

def predict_news(text):
    out = classifier(text)[0]
    return {label_map.get(out['label'], out['label']): float(out['score'])}

demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=4, placeholder='Enter a news sentence...'),
    outputs=gr.Label(num_top_classes=1),
    title='AG News Classifier (Hugging Face Fine-Tuned Model)'
)

print('Run demo.launch() in next cell to start deployment UI.')

In [ ]:
# demo.launch()

## Optional: Push to Hugging Face Hub
Uncomment and run after setting your Hugging Face token.

from huggingface_hub import login
login(token='YOUR_HF_TOKEN')
trainer.push_to_hub('agnews-distilbert-62')

## Result
An open-source Hugging Face model was fine-tuned, evaluated, saved locally, and prepared for deployment using Gradio.